# Cross-hospital CNN experiment
Run the cells in order in a CPU Colab runtime. When prompted, select TB_CNN_Submission.zip or a ZIP downloaded from the team GitHub repository after the cnn_study folder has been added. The results are written to a new directory for each run.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, tempfile, zipfile
from google.colab import files

outer = Path("/content/TB_CNN_Submission.zip")
if not outer.exists():
    uploaded = files.upload()
    outer = Path(next(iter(uploaded)))
    del uploaded
run = Path(tempfile.mkdtemp(prefix="cnn_rerun_", dir="/content"))
with zipfile.ZipFile(outer) as archive:
    entry = next((m for m in archive.namelist()
                  if Path(m).parts[-2:] == ("cnn_study", "experiment.py")), None)
    if entry is None:
        raise ValueError("Choose the submission ZIP or the team repository ZIP containing cnn_study.")
    folder = Path(entry).parent
    for member in archive.infolist():
        path = Path(member.filename)
        if member.is_dir() or not path.is_relative_to(folder):
            continue
        relative = path.relative_to(folder)
        if ".." in relative.parts:
            raise ValueError("Unexpected ZIP path")
        destination = run / "cnn_study" / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, destination.open("wb") as target:
            shutil.copyfileobj(source, target)
project = run / "cnn_study"
assert (project / "experiment.py").exists(), "Choose the main submission ZIP."
os.environ["OPENBLAS_NUM_THREADS"] = "1"
print(project)

## Recover the original split
The image manifest is included. The loader checks the hash of every radiograph against the previous run.

In [ ]:
try:
    import kagglehub
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "kagglehub"], check=True)
    import kagglehub
source = kagglehub.dataset_download("nikhilpandey360/chest-xray-masks-and-labels/versions/1")
subprocess.run([sys.executable, "prepare_data.py", source, "--out", str(run/"images.npz")], cwd=project, check=True)

## Fit the full and ablated networks
Both variants get 20 epochs with the same initialization and minibatches. Only the training-time dropout probability changes.

In [ ]:
subprocess.run([sys.executable, "experiment.py", str(run/"images.npz"), "--out", str(run/"repeat_results")], cwd=project, check=True)

## Inspect the external test and its uncertainty

In [ ]:
import json
import pandas as pd
repeat = run / "repeat_results"
scores = pd.read_csv(repeat / "cnn_metrics.csv")
display(scores.query("rule == 'selected'")[["model","split","TP","FN","TN","FP","sensitivity","specificity","auc"]])
display(pd.read_csv(repeat / "paired_differences.csv"))
summary = json.loads((repeat/"summary.json").read_text())
print("Exact McNemar p:", summary["mcnemar_p"])

## Download repeat-run predictions and weights

In [ ]:
files.download(shutil.make_archive(str(run/"repeat_results"), "zip", repeat))